# 01 · Data Loading, Cleaning & Merge
## Austin Animal Center — Intakes + Outcomes → one analysis-ready table

**Where this fits.** This is the first notebook in the project pipeline:
`01_cleaning` (this file) → `02_eda` → `03_modeling`. It turns the two raw historical
exports into a single cleaned, merged dataset that every later notebook reads from
`../data/processed/df_full_merged.csv`.

**Inputs.** Two local full-history CSV exports (2013-10-01 → 2025-05-05) downloaded from
the Austin Open Data Portal:
- `Austin_Animal_Center_Intakes_…csv` (~173.8k rows) — one row per **intake** event
- `Austin_Animal_Center_Outcomes_…csv` (~173.8k rows) — one row per **outcome** event

**What this notebook does.**
1. **Rename & trim columns** — align both exports to a shared `snake_case` schema and drop columns not needed downstream.
2. **Normalize & validate dates** — convert mixed-timezone ISO datetimes to date-only (YYYY-MM-DD) and drop rows with impossible `date_of_birth` values (DOB after `outcome_date`).
3. **Merge** — backward `merge_asof` intakes ← outcomes on `animal_id`, matching each outcome to its most recent prior intake → one row per outcome event.
4. **Clean the merge** — drop orphan outcomes with no matching intake (~0.5%), deduplicate, and drop outcomes that re-claim the same intake.
5. **Parse sex & breed** — split `sex_upon_intake` into `sex` + the non-leaky `is_previously_spayed_neutered` (known at intake time), and split `breed` into `primary_breed` + `is_mix`; drop the raw source columns.
6. **Clean color** — split `color` on "/" into `primary_color` / `secondary_color`, then route `primary_color` through dog- vs cat-specific maps to derive a coarse base color and `pattern`.
7. **Engineer baseline features** — `length_of_stay_days`, `age_at_intake_days` (negatives dropped), the **primary target `is_long_stay`** (stay > 30 days, AAC's official long-stay definition), the now comparison-only `is_adopted`, the multi-class `outcome_category`, and intake-date temporal features (`intake_year`, `intake_month`, `intake_weekday`).
8. **Append still-in-shelter long-stay animals (Step 8b)** — the outcome-keyed merge omits animals with no outcome yet, so the longest *current* stays are missing. Recover the intake-only records that have **already** stayed > 30 days **and** entered within the last 12 months, adding them as confirmed `is_long_stay = 1` rows (`outcome_category = 'still in shelter'`, censored `length_of_stay_days`).
9. **Save** — reorder to the canonical column layout, filter to **Dog & Cat**, and write `../data/processed/df_full_merged.csv`.

**Output.** `df_full_merged.csv` — **Dog & Cat** only (**162,932 rows**, including **467** appended still-in-shelter long-stay rows) for analysis and modeling.

**Targets.**
- **Binary (primary) — `is_long_stay`**: 1 = length of stay **> 30 days** (Austin Animal Center's official long-stay definition), else 0. The modeling goal is now to flag animals likely to occupy shelter space long-term so staff can intervene early.
- **`is_adopted`** (1 = Adopted incl. `Rto-Adopt`, else 0) — **no longer a modeling target**; demoted to a comparison-only derived column. Kept in the output, not deleted.

**Leakage caution (set here, enforced in `03_modeling`).** Because the target is now `is_long_stay`, every outcome-derived column is a leakage source and must never be a model feature: `outcome_type`, `outcome_subtype`, `outcome_date`, `length_of_stay_days`, and — newly — `is_adopted` and `outcome_category` (both derived from the outcome). **Note:** `01_cleaning` does **not** drop these; it only records the role change. The actual removal-from-features happens in `03_modeling`. `length_of_stay_days` stays a legitimate derived column here — it is the source of `is_long_stay`.

## Setup


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

pandas  3.0.0
numpy   2.4.1


## Data Loading

Local copies of the full historical exports (2013-10-01 to 2025-05-05) downloaded
from the Austin data portal UI.

In [ ]:
FULL_OUTCOMES_PATH = Path('../data/raw_dataset/Austin_Animal_Center_Outcomes_(10_01_2013_to_05_05_2025)_20260523.csv')
FULL_INTAKES_PATH  = Path('../data/raw_dataset/Austin_Animal_Center_Intakes_(10_01_2013_to_05_05_2025)_20260523.csv')

df_full_outcomes = pd.read_csv(FULL_OUTCOMES_PATH)
df_full_intakes  = pd.read_csv(FULL_INTAKES_PATH)

print('=== FULL OUTCOMES EXPORT ===')
df_full_outcomes.info()
print('\n--- head(3) ---')
display(df_full_outcomes.head(3))

print('\n=== FULL INTAKES EXPORT ===')
df_full_intakes.info()
print('\n--- head(3) ---')
df_full_intakes.head(3)

=== FULL OUTCOMES EXPORT ===
<class 'pandas.DataFrame'>
RangeIndex: 173775 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Animal ID         173775 non-null  str  
 1   Date of Birth     173775 non-null  str  
 2   Name              123991 non-null  str  
 3   DateTime          173775 non-null  str  
 4   MonthYear         173775 non-null  str  
 5   Outcome Type      173729 non-null  str  
 6   Outcome Subtype   79660 non-null   str  
 7   Animal Type       173775 non-null  str  
 8   Sex upon Outcome  173774 non-null  str  
 9   Age upon Outcome  173766 non-null  str  
 10  Breed             173775 non-null  str  
 11  Color             173775 non-null  str  
dtypes: str(12)
memory usage: 15.9 MB

--- head(3) ---


,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green



=== FULL INTAKES EXPORT ===
<class 'pandas.DataFrame'>
RangeIndex: 173812 entries, 0 to 173811
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Animal ID         173812 non-null  str  
 1   Name              123821 non-null  str  
 2   DateTime          173812 non-null  str  
 3   MonthYear         173812 non-null  str  
 4   Found Location    173812 non-null  str  
 5   Intake Type       173812 non-null  str  
 6   Intake Condition  173812 non-null  str  
 7   Animal Type       173812 non-null  str  
 8   Sex upon Intake   173811 non-null  str  
 9   Age upon Intake   173812 non-null  str  
 10  Breed             173812 non-null  str  
 11  Color             173812 non-null  str  
dtypes: str(12)
memory usage: 15.9 MB

--- head(3) ---


,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
0,A521520,Nina,10/01/2013 07:51:00 AM,October 2013,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,7 years,Border Terrier/Border Collie,White/Tan
1,A664235,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
2,A664236,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White


## Clean & Merge Full-Export CSVs into One Frame

Align the 173k historical CSV rows to a snake_case schema, then backward-asof
merge intakes ← outcomes so each outcome carries its triggering intake's
context. Output: `df_full_merged` — one row per outcome event (plus the
still-in-shelter intake-only rows appended later in Step 8b).

### Step 1: change the column names into snake_schema for further data cleaning

In [4]:
# --- outcomes side: rename to target schema ---
o = df_full_outcomes.rename(columns={
    'Animal ID':       'animal_id',
    'Name':            'name',
    'DateTime':        'outcome_date',
    'Outcome Type':    'outcome_type',
    'Date of Birth':   'date_of_birth',
    'Outcome Subtype': 'outcome_subtype',
})

o = o.drop(columns=['Sex upon Outcome', 'MonthYear' ,'Age upon Outcome', 'Breed','Animal Type', 'Color']) 

# --- intakes side: rename to target schema ---
i = df_full_intakes.rename(columns={
    'Animal ID':        'animal_id',
    'DateTime':         'intake_date',
    'Found Location':   'found_location',
    'Intake Type':      'intake_reason',
    'Intake Condition': 'intake_health_condition',
    'Breed':            'breed',
    'Animal Type':      'animal_type',
    'Color':            'color',       
    'Sex upon Intake':  'sex_upon_intake',
    'Age upon Intake':  'age_upon_intake_str',   # kept: used to age the appended still-in-shelter rows (Step 8b)
    'Name':             'name_intake',           # kept under a distinct name (outcomes side supplies 'name')
})

# Only MonthYear is redundant. 'age_upon_intake_str' and 'name_intake' are retained on the
# intakes frame `i` for the still-in-shelter append (Step 8b). The outcome-keyed merge in
# Step 3 selects an explicit intake_cols subset, so these extra columns don't affect it.
i = i.drop(columns=['MonthYear'])

print(f'After rename — outcomes: {o.shape}, intakes: {i.shape}')
print('Outcomes cols:', list(o.columns))
print('Intakes  cols:', list(i.columns))

After rename — outcomes: (173775, 6), intakes: (173812, 11)
Outcomes cols: ['animal_id', 'date_of_birth', 'name', 'outcome_date', 'outcome_type', 'outcome_subtype']
Intakes  cols: ['animal_id', 'name_intake', 'intake_date', 'found_location', 'intake_reason', 'intake_health_condition', 'animal_type', 'sex_upon_intake', 'age_upon_intake_str', 'breed', 'color']


### Step 2: Normalize & Validate dates
Convert mixed ISO datetimes to date-only (YYYY-MM-DD), stripping timezone offsets. Drop any rows where date_of_birth is after outcome_date (impossible).

In [5]:
# Parse mixed ISO datetimes into date-only (YYYY-MM-DD)
# Reasons:
# - Source strings contain both timezone-aware (e.g. 2013-12-02T00:00:00-05:00) and naive ISO strings (e.g. 2013-10-01T09:31:00).
# - If pandas parses a Series as tz-aware, the naive strings can become NaT.
# - To avoid that, strip the timezone offset suffix, parse, then normalize to date.

def _parse_date_only(series: pd.Series) -> pd.Series:
    """Return a date-only Series (datetime64[ns]) from mixed ISO datetime strings."""
    series = series.astype(str).str.strip()
    # Remove timezone offsets like -05:00 or +00:00 so both offset and naive strings parse
    series = series.str.replace(r'([+-]\d{2}):(\d{2})$', '', regex=True)
    # Parse; errors='coerce' turns unparseable values into NaT, .dt.normalize() keeps date only
    return pd.to_datetime(series, errors='coerce').dt.normalize()

# Apply parsing to columns (works whether the column currently holds raw strings
# or Timestamp objects represented as strings)
o['outcome_date'] = _parse_date_only(o['outcome_date'])
o['date_of_birth'] = _parse_date_only(o['date_of_birth'])
i['intake_date'] = _parse_date_only(i['intake_date'])

# DOB after outcome is impossible — drop these rows (compare dates only).
# NaT DOBs (parse failures) compare False, so they are kept, not dropped.
before = len(o)
o = o[~(o['date_of_birth'] > o['outcome_date'])].copy()
dropped_dob = before - len(o)

# Audits
print(f'Dropped impossible DOBs (DOB > outcome): {dropped_dob:,}')
print(f'Null outcome dates: {o["outcome_date"].isna().sum():,}')
print(f'Null intake dates: {i["intake_date"].isna().sum():,}')

Dropped impossible DOBs (DOB > outcome): 33
Null outcome dates: 0
Null intake dates: 0


### Step 3: Merge intakes ← outcomes

Backward `merge_asof` on `animal_id`, matching each outcome to the most recent
prior intake (`direction='backward'`), so every outcome row carries its
triggering intake's context. Output: `df_full_merged` — one row per outcome event.
(Unmatched outcomes and duplicates are dropped in Steps 4–5; still-in-shelter
intakes that never got an outcome are appended back in Step 8b.)

In [6]:
intake_cols = [
    'animal_id', 
    'intake_date', 
    'found_location', 
    'intake_reason', 
    'intake_health_condition', 
    'animal_type', 
    'sex_upon_intake', 
    'breed', 
    'color']

intakes_for_merge = (
    i[intake_cols]
    .sort_values('intake_date')
)
outcomes_sorted = o.sort_values('outcome_date')

df_full_merged = pd.merge_asof(
    outcomes_sorted,
    intakes_for_merge,
    by='animal_id',
    left_on='outcome_date',
    right_on='intake_date',
    direction='backward',
)


### Step 4: Drop unmatched outcomes and deduplicate merged data

Filter out merged outcome rows that have no matched prior `intake_date`, then remove duplicate rows from `df_full_merged`. Print the number of rows dropped and audit the final row count and missing intake dates.

In [7]:
# drop orphans with missing intake date (~0.5%)
before = len(df_full_merged)
df_full_merged = df_full_merged[df_full_merged['intake_date'].notna()].copy()
dropped = before - len(df_full_merged)
print(f'Dropped {dropped:,} rows missing intake_date '
      f'({dropped / before * 100:.2f}%) — {len(df_full_merged):,} rows remain')

# drop duplicates
print(f'Drop duplicates:             {df_full_merged.duplicated().sum():,}')
df_full_merged.drop_duplicates(inplace=True)

# Audits
print(f'Row parity (merged vs outcomes): {len(df_full_merged):,} vs {len(o):,}')
print(f'Missing intake_date:             {df_full_merged["intake_date"].isna().sum():,}')

Dropped 924 rows missing intake_date (0.53%) — 172,818 rows remain
Drop duplicates:             47
Row parity (merged vs outcomes): 172,771 vs 173,742
Missing intake_date:             0


### Step 5: Drop outcomes that re-claim the same intake

Filter out merged outcome rows that matched the same `intake_date`, keeping only the first record.

In [8]:
before = len(df_full_merged)
# A single intake for the same animal can map to only one outcome;
# if shared by multiple outcomes, keep the earliest outcome_date (closest to that shelter stay)
df_full_merged = (
    df_full_merged
    .sort_values('outcome_date')
    .drop_duplicates(subset=['animal_id', 'intake_date'], keep='first')
)
dropped = before - len(df_full_merged)
print(f'Removed outcomes re-claiming the same intake: {dropped:,} rows '
      f'({dropped/before*100:.2f}%)')

Removed outcomes re-claiming the same intake: 274 rows (0.16%)


### Merge audit: intake-only records (right-censoring check)

The backward `merge_asof` is keyed on **outcomes**, so `df_full_merged` only contains
animals that already have an outcome. Any intake whose `(animal_id, intake_date)` pair
is **not** among the claimed pairs is an *intake-only* record — the animal entered the
shelter but has no recorded outcome.

This is a diagnostic only (nothing is added to or removed from the output). It checks
**when** those records occur, to confirm they are end-of-window right-censoring (animals
still in care at the export date) rather than random data loss across the full history.

In [9]:
# --- Diagnostic (not written to output): intakes that never matched an outcome ---
# After Steps 3-5, df_full_merged is one row per outcome, each carrying the
# (animal_id, intake_date) pair it claimed. A Dog/Cat intake whose pair is NOT in
# that set has no recorded outcome. Reuse `i` (the raw intakes frame from Step 1).
claimed_pairs = set(zip(df_full_merged['animal_id'], df_full_merged['intake_date']))

intakes_dc = (
    i[i['animal_type'].isin(['Dog', 'Cat'])]
    .dropna(subset=['intake_date'])
    .copy()
)
intakes_dc['has_outcome'] = [
    pair in claimed_pairs
    for pair in zip(intakes_dc['animal_id'], intakes_dc['intake_date'])
]
orphans = intakes_dc[~intakes_dc['has_outcome']]

n_tot, n_orph = len(intakes_dc), len(orphans)
latest_intake = intakes_dc['intake_date'].max()
export_cutoff = o['outcome_date'].max()   # data window end (2025-05-05); last intake is a day earlier
print(f'Dog/Cat intakes (valid date):     {n_tot:,}')
print(f'Intake-only (no outcome) records: {n_orph:,}  ({n_orph / n_tot * 100:.2f}%)')
print('By species: ' + ', '.join(f'{k} {v:,}'
      for k, v in orphans['animal_type'].value_counts().items()))
print(f'Latest intake in file:            {latest_intake.date()}')
print(f'Data window end (export cutoff):  {export_cutoff.date()}')
print('=' * 60)

# Orphan rate by intake year — flat historical baseline vs the end-of-window spike
by_year = pd.DataFrame({
    'orphans': orphans.groupby(orphans['intake_date'].dt.year).size(),
    'intakes': intakes_dc.groupby(intakes_dc['intake_date'].dt.year).size(),
})
by_year['orphan_rate_%'] = (by_year['orphans'] / by_year['intakes'] * 100).round(1)
print('Orphan rate by intake year:')
print(by_year.fillna(0).astype({'orphans': int}).to_string())

Dog/Cat intakes (valid date):     163,932
Intake-only (no outcome) records: 1,243  (0.76%)
By species: Dog 787, Cat 456
Latest intake in file:            2025-05-04
Data window end (export cutoff):  2025-05-05
Orphan rate by intake year:
             orphans  intakes  orphan_rate_%
intake_date                                 
2013               4     3943           0.10
2014              25    17505           0.10
2015              28    17812           0.20
2016              40    16566           0.20
2017              34    16429           0.20
2018              44    16023           0.30
2019              36    18810           0.20
2020              18     8869           0.20
2021              25    11341           0.20
2022              21    11178           0.20
2023              25    10686           0.20
2024             144    11236           1.30
2025             799     3534          22.60


### Step 6: Parse sex, spay/neuter and breed fields

From `sex_upon_intake` (known at intake time, so non-leaky) derive `sex` and the
boolean `is_previously_spayed_neutered`. The outcome-side sex/spay column was already
dropped back in Step 1 because it's target leakage (shelters spay/neuter as part of the
adoption process, so that flag is caused by the outcome). Split `breed` into
`primary_breed` + the `is_mix` flag (1 = mixed/crossbreed, i.e. has " Mix" or "/"),
then drop the raw `sex_upon_intake` and `breed` source columns.

In [10]:
SEX_UPON_MAP = {
    'Neutered Male': ('Male',    'Yes'),
    'Spayed Female': ('Female',  'Yes'),
    'Intact Male':   ('Male',    'No'),
    'Intact Female': ('Female',  'No'),
    'Unknown':       ('Unknown', 'Unknown'),
}


def _split_sex(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    mapped = series.map(SEX_UPON_MAP)
    sex  = mapped.map(lambda t: t[0] if isinstance(t, tuple) else 'Unknown')
    spay = mapped.map(lambda t: t[1] if isinstance(t, tuple) else 'Unknown')
    return sex, spay

# split breed into primary breed and is_mix (1/0)
def parse_breed(b):
    if pd.isna(b):
        return (np.nan, pd.NA)
    b = str(b).strip()
    # 1 = mixed/crossbreed (has " Mix" or a "/"), 0 = single breed
    is_mix = 1 if (('mix' in b.lower()) or ('/' in b)) else 0
    # Take first breed token (before "/" or " Mix")
    primary = b.split('/')[0].replace(' Mix', '').replace(' mix', '').strip()
    return (primary, is_mix)

# --- split Sex upon Intake -> sex + is_previously_spayed_neutered ---
df_full_merged['sex'], _ = _split_sex(df_full_merged['sex_upon_intake'])
_, spay_intake = _split_sex(df_full_merged['sex_upon_intake'])
df_full_merged['is_previously_spayed_neutered'] = spay_intake.map({'Yes': 1, 'No': 0}).astype('Int64')
df_full_merged = df_full_merged.drop(columns=['sex_upon_intake'])


# --- breed -> primary breed + is_mix (1/0) ---
parsed = df_full_merged['breed'].apply(parse_breed)
df_full_merged['primary_breed'] = [p[0] for p in parsed]
df_full_merged['is_mix']        = pd.array([p[1] for p in parsed], dtype='Int64')
df_full_merged = df_full_merged.drop(columns=['breed'])


### Step 7: Clean `color` column

Split `color` into `primary_color` and `secondary_color`, then route `primary_color`
through dog- vs cat-specific maps to derive a coarse base color (overwriting
`primary_color`) and a `pattern` column. The split logic is specified separately for
the different color distributions of dogs and cats.

In [11]:
TOPN = 10
COLOR_COL = 'color'   # raw/original color, before to_color_group

# --- tables: top-N raw colors per species ---
for sp in ['Dog', 'Cat']:
    vc = df_full_merged.loc[df_full_merged['animal_type'] == sp, COLOR_COL].value_counts()
    print(f'\n=== {sp}: {vc.size} distinct {COLOR_COL} (showing top {TOPN}) ===')
    print(vc.head(TOPN).to_string())



=== Dog: 400 distinct color (showing top 10) ===
color
Black/White    11114
Brown/White     5334
White           5154
Tan/White       4902
Black           4900
Tan             4073
Brown           3818
Black/Tan       3597
Tricolor        3441
Black/Brown     3405

=== Cat: 327 distinct color (showing top 10) ===
color
Brown Tabby           10631
Black                  8928
Black/White            6190
Brown Tabby/White      5414
Orange Tabby           5106
Tortie                 3150
Calico                 2960
Blue Tabby             2792
Orange Tabby/White     2552
Blue                   2547


In [12]:
def _split_slash(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    parts = series.fillna('').str.split('/', n=1, expand=True)
    primary = parts[0].replace('', np.nan)
# If no value contains '/', split yields a single column. Return an all-NaN
# Series (not a bare scalar) so the secondary_color assignment stays aligned.

    if parts.shape[1] > 1:
        secondary = parts[1].replace('', np.nan)
    else:
        secondary = pd.Series(np.nan, index=series.index)
    return primary, secondary

df_full_merged['primary_color'], df_full_merged['secondary_color'] = _split_slash(df_full_merged['color'])
df_full_merged = df_full_merged.drop(columns=['color'])

In [13]:
# ===== Dog route =====
DOG_COLOR_BASE = {
    'Black':'Black', 'White':'White',
    'Brown':'Brown', 'Chocolate':'Brown', 'Sable':'Brown', 'Liver':'Brown',
    'Tan':'Tan', 'Fawn':'Tan', 'Buff':'Tan', 'Apricot':'Tan', 'Gold':'Tan', 'Yellow':'Tan',
    'Blue':'Blue/Gray', 'Gray':'Blue/Gray', 'Silver':'Blue/Gray',
    'Red':'Red/Orange', 'Orange':'Red/Orange',
    'Cream':'Cream',
    'Tricolor':'Tricolor', 'Tricolored':'Tricolor',
}
DOG_PATTERN_KEYS      = ['Brindle', 'Merle', 'Tick', 'Tiger']
DOG_PATTERN_NORMALIZE = {'Tiger': 'Brindle'}

# ===== Cat route =====
CAT_COLOR_BASE = {
    'Black':'Black', 'White':'White',
    'Blue':'Blue/Gray', 'Gray':'Blue/Gray', 'Silver':'Blue/Gray', 'Lilac':'Blue/Gray',
    'Cream':'Cream', 'Orange':'Orange',
    'Seal':'Point-tint', 'Flame':'Point-tint', 'Lynx':'Point-tint',
    'Brown':'Brown', 'Chocolate':'Brown',                          # plain choc -> Brown; Choc Point -> base Brown + pattern Point
    'Torbie':'Tortie', 'Tortie':'Tortie',         # Tortie family treated as base (see markdown)
    'Calico':'Calico',
}
CAT_PATTERN_KEYS      = ['Tabby', 'Point', 'Tiger', 'Smoke']
CAT_PATTERN_NORMALIZE = {'Tiger': 'Tabby'}

def split_base_pattern(c, color_base, pattern_keys, pattern_normalize):
    """Return (base_color, pattern) using the species-specific maps.
       NaN->('Unknown','Unknown'); no pattern word -> 'Solid';
       pattern word but no base word -> base='None'."""
    if pd.isna(c):
        return ('Unknown', 'Unknown')
    c = str(c).strip()

    pattern = next((p for p in pattern_keys if p in c), 'Solid')
    pattern = pattern_normalize.get(pattern, pattern)

    base_str = c
    for p in pattern_keys:
        base_str = base_str.replace(p, '')
    # also strip any base-words that are really pattern-family labels (Tortie/Calico/Torbie)
    # so the remaining token is the true tint; keep them only if nothing else remains.
    base_str = base_str.strip()
    base = color_base.get(base_str.split()[0], 'Other') if base_str else 'None'
    return (base, pattern)

def route_split(row):
    if row['animal_type'] == 'Dog':
        return split_base_pattern(row['primary_color'], DOG_COLOR_BASE, DOG_PATTERN_KEYS, DOG_PATTERN_NORMALIZE)
    elif row['animal_type'] == 'Cat':
        return split_base_pattern(row['primary_color'], CAT_COLOR_BASE, CAT_PATTERN_KEYS, CAT_PATTERN_NORMALIZE)
    return ('Unknown', 'Unknown')

_parsed = df_full_merged.apply(route_split, axis=1)
df_full_merged['primary_color'] = _parsed.map(lambda t: t[0])
df_full_merged['pattern']    = _parsed.map(lambda t: t[1])

print('Added primary_color / pattern via dog & cat routes.')

Added primary_color / pattern via dog & cat routes.


### Step 8: Compute baseline features
Add baseline features
- **`length_of_stay_days`** — `outcome_date − intake_date`
- **`is_long_stay`** — **primary target**, 1 if `length_of_stay_days > 30`, else 0
- **`age_at_intake_days`** — `intake_date − date_of_birth` (rows with negative age dropped)
- **`is_adopted`** — comparison-only, 1 if adopted else 0
- **`outcome_category`** - combine into 'adopted', 'return to owner', 'transfer', 'died', 'other'
- **`intake_year`** - year of intake date
- **`intake_month`** - month of intake date
- **`intake_weekday`** - intake day of week

In [14]:
# Derived column- 'length_of_stay_days'
df_full_merged['length_of_stay_days'] = (
    df_full_merged['outcome_date'] - df_full_merged['intake_date']
).dt.days
print(f'Negative length of stay: {(df_full_merged["length_of_stay_days"] < 0).sum():,} days')
print(f'Median length of stay:   {df_full_merged["length_of_stay_days"].median():.0f} days')
print('='*60)

# Derived column- 'is_long_stay' (PRIMARY TARGET): AAC official long-stay = stay over 30 days (strict >)
LONG_STAY_THRESHOLD = 30
df_full_merged['is_long_stay'] = (
    df_full_merged['length_of_stay_days'] > LONG_STAY_THRESHOLD
).astype(int)
print('is_long_stay rate by animal_type:')
print(df_full_merged.groupby('animal_type')['is_long_stay'].mean())
print('='*60)

# Derived column- 'age_at_intake_days'
df_full_merged['age_at_intake_days'] = (
    df_full_merged['intake_date'] - df_full_merged['date_of_birth']
).dt.days
before = len(df_full_merged)
df_full_merged = df_full_merged[~(df_full_merged['age_at_intake_days'] < 0)].copy()
dropped = before - len(df_full_merged)
df_full_merged['age_at_intake_days'] = df_full_merged['age_at_intake_days'].astype('Int64')
print(f'Dropped {dropped:,} rows with negative age_at_intake_days '
      f'({dropped / before * 100:.2f}%) — {len(df_full_merged):,} rows remain')
print('='*60)


# Derived column- 'is_adopted'
# Group the 11 raw outcome_type values into 5 modeling categories.
OUTCOME_CATEGORY_MAP = {
    'Adoption':        'adopted',
    'Rto-Adopt':       'adopted',
    'Return to Owner': 'return to owner',
    'Transfer':        'transfer',
    'Euthanasia':      'died',
    'Disposal':        'died',
    'Died':            'died',
}

# 'is_adopted' is created for another modeling (1 if adopted or Rto-Adopt, else 0)
df_full_merged['is_adopted'] = df_full_merged['outcome_type'].isin(['Adoption', 'Rto-Adopt']).astype(int)
df_full_merged['outcome_category'] = df_full_merged['outcome_type'].map(OUTCOME_CATEGORY_MAP).fillna('other')

# Derived temporal columns
df_full_merged['intake_year'] = df_full_merged['intake_date'].dt.year
df_full_merged['intake_month'] = df_full_merged['intake_date'].dt.month
df_full_merged['intake_weekday'] = df_full_merged['intake_date'].dt.weekday

Negative length of stay: 0 days
Median length of stay:   6 days
is_long_stay rate by animal_type:
animal_type
Bird        0.07
Cat         0.25
Dog         0.16
Livestock   0.45
Other       0.04
Name: is_long_stay, dtype: float64
Dropped 191 rows with negative age_at_intake_days (0.11%) — 172,306 rows remain


### Step 8b: Append still-in-shelter long-stay animals

The outcome-keyed merge (Steps 3–5) yields **one row per outcome**, so animals that entered
but have **no outcome yet** (still in the shelter at the export cutoff) are absent — yet those
are exactly the longest *current* stays. Here we recover the ones that have **already** stayed
long enough to be confirmed long-stay, appending them as extra rows (the outcome-based pipeline
above is left untouched):

- **Group A (append):** orphan Dog/Cat intakes that are **both** already long-stay
  (`EXPORT_DATE − intake_date > 30` days) **and recent** (intake within the last 12 months,
  `intake_date ≥ EXPORT_DATE − 12 months`) → confirmed `is_long_stay = 1`. Their
  `length_of_stay_days` is a **lower bound** (right-censored), `outcome_*`/`date_of_birth` are
  unknown (`'still in shelter'` / `NaT`), and `age_at_intake_days` is parsed from the
  intake-side age string (`age_upon_intake_str`).
- **Dropped:** orphan intakes ≤ 30 days old (outcome not known yet — genuine end-of-window
  censoring) **and** orphan intakes older than 12 months. The latter are the flat historical
  baseline of missing-outcome data gaps (≈25/year back to 2013); with no outcome logged years
  ago they are data-quality artifacts, not animals literally still in the shelter, so the
  12-month bound excludes them to keep the `'still in shelter'` label meaningful.

We reuse the **validated** orphan set from the Merge-audit diagnostic above (its claimed set is
taken right after Step 5, so the negative-age outcome rows dropped in Step 8 are not mistaken for
orphans). The enlarged `df_full_merged` then flows unchanged into Step 9's reorder / Dog-Cat
filter / save.

In [15]:
# ===== APPEND still-in-shelter long-stay animals (Group A) =====
# Reuse the VALIDATED orphan set from the Merge-audit diagnostic above (`orphans`):
# Dog/Cat intakes whose (animal_id, intake_date) pair was never claimed by an outcome.
# NOTE: we deliberately do NOT recompute the claimed set here — df_full_merged has since
# had its negative-age outcome rows dropped (Step 8), which would be misread as orphans.
EXPORT_DATE    = pd.Timestamp('2025-05-04')                 # max intake_date / export cutoff
RECENCY_CUTOFF = EXPORT_DATE - pd.DateOffset(months=12)     # only intakes within the last 12 months
                                                            # count as genuinely "still in shelter"

orphans = orphans.copy()
orphans['days_since_intake'] = (EXPORT_DATE - orphans['intake_date']).dt.days

# Group A: already long-stay (>30d) AND recent enough to still plausibly be in care (<=12mo)
group_a = orphans[(orphans['days_since_intake'] > 30) &
                  (orphans['intake_date'] >= RECENCY_CUTOFF)].copy()

# Everything else is dropped, for two distinct reasons:
n_drop_recent_censored = int((orphans['days_since_intake'] <= 30).sum())        # outcome not known yet
n_drop_old_gap         = int((orphans['intake_date'] < RECENCY_CUTOFF).sum())   # historic missing-outcome gaps

# --- age from the intake age string ("7 years"/"2 months"/"1 week"/"3 days") ---
import re
_AGE_UNIT_DAYS = {'year': 365, 'month': 30, 'week': 7, 'day': 1}
def _age_str_to_days(s):
    if pd.isna(s):
        return pd.NA
    m = re.match(r'(\d+)\s*(year|month|week|day)', str(s).strip().lower())
    return int(m.group(1)) * _AGE_UNIT_DAYS[m.group(2)] if m else pd.NA
group_a['age_at_intake_days'] = group_a['age_upon_intake_str'].map(_age_str_to_days).astype('Int64')

# --- parse sex / spay-neuter / breed / color with the SAME functions used in Steps 6-7 ---
group_a['sex'], _spay = _split_sex(group_a['sex_upon_intake'])
group_a['is_previously_spayed_neutered'] = _spay.map({'Yes': 1, 'No': 0}).astype('Int64')

_parsed_breed = group_a['breed'].apply(parse_breed)
group_a['primary_breed'] = [p[0] for p in _parsed_breed]
group_a['is_mix']        = pd.array([p[1] for p in _parsed_breed], dtype='Int64')

group_a['primary_color'], group_a['secondary_color'] = _split_slash(group_a['color'])
_parsed_color = group_a.apply(route_split, axis=1)
group_a['primary_color'] = _parsed_color.map(lambda t: t[0])
group_a['pattern']       = _parsed_color.map(lambda t: t[1])

# --- outcome-side / derived fields for still-in-shelter rows ---
group_a['date_of_birth']       = pd.Series(pd.NaT, index=group_a.index, dtype='datetime64[us]')
group_a['outcome_date']        = pd.Series(pd.NaT, index=group_a.index, dtype='datetime64[us]')
group_a['length_of_stay_days'] = (EXPORT_DATE - group_a['intake_date']).dt.days   # lower bound (censored)
group_a['is_long_stay']        = 1
group_a['is_adopted']          = 0
group_a['outcome_category']    = 'still in shelter'
group_a['outcome_type']        = 'still in shelter'
group_a['outcome_subtype']     = 'still in shelter'
group_a['name']                = group_a['name_intake']            # intake-side name
group_a['intake_year']         = group_a['intake_date'].dt.year
group_a['intake_month']        = group_a['intake_date'].dt.month
group_a['intake_weekday']      = group_a['intake_date'].dt.weekday

# align to df_full_merged's columns exactly, then append
group_a = group_a.reindex(columns=df_full_merged.columns)
df_full_merged = pd.concat([df_full_merged, group_a], ignore_index=True)

print(f'Appended Group A (still-in-shelter, >30d & within 12mo): {len(group_a):,} rows')
print(f'Dropped, recent & censored (<=30d):                      {n_drop_recent_censored:,} rows')
print(f'Dropped, >12mo old (missing-outcome data gaps):          {n_drop_old_gap:,} rows')
print(f'New df_full_merged shape: {df_full_merged.shape}')

Appended Group A (still-in-shelter, >30d & within 12mo): 467 rows
Dropped, recent & censored (<=30d):                      456 rows
Dropped, >12mo old (missing-outcome data gaps):          320 rows
New df_full_merged shape: (172773, 26)


In [16]:
# --- Validation: the appended still-in-shelter rows are well-formed long-stays ---
appended = df_full_merged[df_full_merged['outcome_category'] == 'still in shelter']
assert (appended['is_long_stay'] == 1).all()
assert (appended['length_of_stay_days'] > 30).all()
assert (appended['intake_date'] >= RECENCY_CUTOFF).all()
assert appended['outcome_date'].isna().all()
assert appended['date_of_birth'].isna().all()
assert len(appended) == len(group_a)
print('Group A checks passed:', len(appended), 'rows')

Group A checks passed: 467 rows


### Step 9: Store processed data into csv

In [17]:
# Reorder columns to the requested canonical order
desired_order = [
    'animal_id',
    'name',
    'animal_type',
    'sex',
    'primary_breed',
    'is_mix',
    'date_of_birth',
    'primary_color',
    'pattern',
    'secondary_color',
    'intake_date',
    'intake_reason',
    'found_location',
    'age_at_intake_days',
    'intake_health_condition',
    'is_previously_spayed_neutered',
    'outcome_date',
    'outcome_type',
    'outcome_subtype',
    'length_of_stay_days',
    'is_long_stay',
    'is_adopted',
    'outcome_category',
    'intake_year',
    'intake_month',
    'intake_weekday',
]

existing = [c for c in desired_order if c in df_full_merged.columns]
missing = [c for c in desired_order if c not in df_full_merged.columns]
if missing:
    print('Warning: missing columns not found and will be skipped:', missing)

# Preserve any other columns that are not in the requested list
other_cols = [c for c in df_full_merged.columns if c not in existing]

# Reassign with new column order
df_full_merged = df_full_merged[existing + other_cols]
print('Reordered df_full_merged columns — first columns now:')
print(df_full_merged.columns.tolist()[:len(existing)])

#check unique values of animal_type and filter to contain only dogs and cats for EDA
df_full_merged['animal_type'].unique() 
df_full_merged= df_full_merged[df_full_merged['animal_type'].isin(['Dog', 'Cat'])]

processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

out_path = processed_dir / 'df_full_merged.csv'
df_full_merged.to_csv(out_path, index=False)

print(f'Saved {len(df_full_merged):,} rows to {out_path}')

Reordered df_full_merged columns — first columns now:
['animal_id', 'name', 'animal_type', 'sex', 'primary_breed', 'is_mix', 'date_of_birth', 'primary_color', 'pattern', 'secondary_color', 'intake_date', 'intake_reason', 'found_location', 'age_at_intake_days', 'intake_health_condition', 'is_previously_spayed_neutered', 'outcome_date', 'outcome_type', 'outcome_subtype', 'length_of_stay_days', 'is_long_stay', 'is_adopted', 'outcome_category', 'intake_year', 'intake_month', 'intake_weekday']
Saved 162,932 rows to ../data/processed/df_full_merged.csv


In [18]:
df_full_merged.head()
df_full_merged.info()

<class 'pandas.DataFrame'>
Index: 162932 entries, 0 to 172772
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   animal_id                      162932 non-null  str           
 1   name                           121327 non-null  str           
 2   animal_type                    162932 non-null  str           
 3   sex                            162932 non-null  str           
 4   primary_breed                  162932 non-null  str           
 5   is_mix                         162932 non-null  Int64         
 6   date_of_birth                  162465 non-null  datetime64[us]
 7   primary_color                  162932 non-null  str           
 8   pattern                        162932 non-null  str           
 9   secondary_color                87311 non-null   str           
 10  intake_date                    162932 non-null  datetime64[us]
 11  intake_reason   